# News Embeddings Pipeline — Step by Step

Braço 02: the frozen text corpus → one dense **BERTimbau** vector per article →
mean and dispersion per (epidemiological week, location).

Mirrors `src/pipeline/news/embeddings/` (spec → steps → transform → integrity).
The hypothesis, the evidence and the counter-evidence are in
[`notes/02-embeddings-densos.md`](../notes/02-embeddings-densos.md); the methods
write-up and the measured numbers are in
[`src/pipeline/news/README.md`](../src/pipeline/news/README.md) §3.

This is one of three **competing representations** of the same corpus — not a
layer on top of braço 01. All three read the same frozen table and publish the
same grain, so the comparison isolates the representation and nothing else. See
[`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md).


## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.news.embeddings import steps as emb_steps
from pipeline.news.embeddings.integrity import run_all as embedding_checks
from pipeline.news.embeddings.spec import (
    ARTICLE_ID, EMB_MEAN, EMB_STD, EMBEDDING, EMBEDDING_DIM, EW, LOCATION,
    MAX_TOKENS, MODEL_NAME, MODEL_REVISION_SHA, N_ARTICLES, N_TOKENS,
)
from pipeline.news.embeddings.transform import select_articles
from pipeline.news.lexicon import steps as lex_steps
from pipeline.news.text import steps as text_steps

pl.Config.set_tbl_rows(14)
pl.Config.set_fmt_str_lengths(70)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "news"
LEXICON_PATH = PROJECT_ROOT / "data" / "news" / "reference" / "lexico_dengue.yaml"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"

NEWS_DIR = PROJECT_ROOT / "data" / "pipeline" / "news"
CORPUS_PATH = NEWS_DIR / "EBC_dengue_text.parquet"
ARTICLES_PATH = NEWS_DIR / "EBC_dengue_embeddings_article.parquet"
POOLED_PATH = NEWS_DIR / "EBC_dengue_embeddings_EW.parquet"
SINAN_PATH = PROJECT_ROOT / "data" / "epidemiological" / "SINAN" / "SINAN_dengue_cases.parquet"

FIXTURES.exists(), LEXICON_PATH.exists(), FU_PATH.exists()

## Literature checklist (what this notebook must prove)

1. The encoder is **frozen** and pinned to a commit sha, so the vectors are
   reproducible and "which BERTimbau?" has an answer that does not move.
2. Pooling is the **masked mean** of the last hidden state — not `[CLS]`, not the
   pooler head — and padding does not change a vector.
3. Articles above `MAX_TOKENS` (512) are truncated, and `n_tokens` makes that
   visible per article instead of silent.
4. `emb_std` separates a week of one repeated story from a week of five unrelated
   ones, at equal volume.
5. The article set is the **same** one braço 01 sees: the language and
   dengue-term gates are common ground under protocol 00, not representation.
6. A week with no article is **absent**, not a zero vector.
7. The fixture is five pages, so it cannot show truncation at scale, the weekly
   coverage or the Spanish slice. Those need the real corpus, and the
   **Optional — the real corpus** section below is where they are measured.


## Step 1 — the frozen encoder

`load_encoder` is the only place the checkpoint is named, and it names the
*revision* as well as the model. A branch would move under the artifact, and an
embedding computed last month would stop being reproducible.

It needs the optional extra, which is deliberately not a core dependency —
`torch` is ~800 MB and nothing else in the repository imports it:

```bash
uv sync --extra embeddings
```

The cell below falls back to a deterministic stand-in when the extra is missing,
so the aggregation steps still run. **When it falls back, the vectors are not
BERTimbau** and nothing downstream of them is a result.


In [ ]:
try:
    encoder = emb_steps.load_encoder(MODEL_NAME, MODEL_REVISION_SHA)
    print(f"loaded {encoder.model_id}@{encoder.model_revision[:8]} — {encoder.dim}-wide")
except emb_steps.MissingExtraError as error:
    import zlib

    class _StandIn:
        """Deterministic text → unit vector. NOT BERTimbau."""

        model_id = "stand-in/not-bertimbau"
        model_revision = "0" * 40
        dim = EMBEDDING_DIM

        def encode(self, texts):
            vectors = []
            for text in texts:
                rng = np.random.default_rng(zlib.crc32(text.encode("utf-8")))
                vector = rng.standard_normal(self.dim)
                vectors.append(vector / np.linalg.norm(vector))
            return emb_steps.Encoded(
                vectors=np.stack(vectors).astype(np.float32),
                n_tokens=[len(t.split()) for t in texts],
            )

    encoder = _StandIn()
    print(error)
    print(">>> FALLBACK ACTIVE: the vectors below are NOT BERTimbau <<<")

print(f"model_id       = {encoder.model_id}")
print(f"model_revision = {encoder.model_revision[:12]}")
print(f"dim            = {encoder.dim}")

## Step 2 — masked mean pooling

Three ways to turn a `(tokens, 768)` matrix into a `(768,)` vector, and the choice
is not cosmetic:

| read | what it is | why not |
|---|---|---|
| `[CLS]` | the token BERT trains for next-sentence prediction | it is a sentence-pair signal, not a summary |
| pooler head | a linear layer on top of `[CLS]` | discarded by every sentence-embedding recipe |
| **masked mean** | the average over the real tokens | **this is what braço 02 uses** |

"Masked" is the load-bearing word. A padded batch carries `[PAD]` positions, and
averaging over them mixes in whatever the model happens to represent `[PAD]` as —
so the same sentence would get a different vector depending on what it was batched
with. The check below encodes one sentence alone and again next to a shorter one:
the vectors must be identical.


In [ ]:
probe = "A dengue avança no interior de São Paulo e a rede pública alerta para o surto."
alone = encoder.encode([probe]).vectors[0]
padded = encoder.encode([probe, "Nota curta."]).vectors[0]

print(f"same text, alone vs padded in a batch of 2")
print(f"  max |difference|         = {np.abs(alone - padded).max():.2e}")
print(f"  norm of the vector       = {np.linalg.norm(alone):.6f}   (L2-normalized)")
print(f"  a padding token is inside the batch: {padded.shape == (encoder.dim,)}")

## Step 3 — one vector per article

The article set is decided *before* the encoder runs, by the same two gates braço
01 applies: drop the ~3.7% of the corpus that is Spanish (Agência Brasil publishes
a Spanish edition), and drop pages that never name a dengue-family disease.

This is not a detail. If one braço sees 4,369 articles and another sees 4,569, the
comparison measures the filter rather than the representation, which is exactly
what protocol 00 exists to prevent.


In [ ]:
corpus = text_steps.build(FIXTURES)
lexicon = lex_steps.load_lexicon(LEXICON_PATH)
matcher = lex_steps.load_uf_matcher(FU_PATH)

kept, dropped = select_articles(corpus, lexicon)
articles = emb_steps.encode_articles(kept, encoder, batch_size=2)

print(f"{corpus.height} fixture pages → {kept.height} kept, {dropped} dropped by the shared gates")
print(f"article table: {articles.height} rows, embedding column {articles.schema[EMBEDDING]}\n")
articles.select(ARTICLE_ID, EW, N_TOKENS, "dim").head()

### Truncation is recorded, not hidden

`n_tokens` is the length the tokenizer produced *before* the `MAX_TOKENS` cut. It
is carried per row because a 34,960-character era-B article and a 163-character
era-C note both become one 768-vector, and only this column says which of them
lost text.


In [ ]:
over, longest = emb_steps.truncation_stats(articles)
print(f"articles above {MAX_TOKENS} tokens: {over}/{articles.height}")
print(f"longest article: {longest} tokens")
articles.select(ARTICLE_ID, N_TOKENS).sort(N_TOKENS, descending=True).head(5)

## Step 4 — the shared scoping

`UfMatcher` is imported from braço 01 on purpose. News is national and the target
is a UF, so the spatial mismatch is a **declared** limitation; re-solving it per
braço would mean the three representations no longer differ only in how they read
the text.

Two consequences worth remembering while reading any per-UF row below: full state
names match case-sensitively (`Pará` vs the preposition `para`), and municipalities
are not resolved — an article that says only "Araçutuba" counts for BR and not for SP.


In [ ]:
locations = emb_steps.scoped_locations(kept, matcher)
locations.head(6)

## Step 5 — pooling to (ew, location)

Every article counts for `BR`; it also counts for each UF it names. One article can
therefore land in several rows of the same week — the same rule braço 01 uses, and
the reason `n_articles` can sum to more than the size of the corpus.

`emb_std` is the **norm** of the per-dimension standard deviation, not the vector:
a 768-wide std column would double the table to say one number the norm already
carries. It is the population std, so a week with a single article is exactly 0 —
"no dispersion observed", which is the honest reading and keeps the column finite.

A week with no article is **absent**, not a zero vector. A zero embedding is a real
point in the space and would read as "news that means nothing" rather than "no
news". `notes/02` asks for a `no_news` indicator; absence is that indicator, and
the consumer left-joins onto the target calendar exactly as it does for braço 01.


In [ ]:
pooled = emb_steps.pool_by_week(articles, locations, encoder.dim)

print(f"{articles.height} articles → {pooled.height} pooled rows over "
      f"{pooled[EW].n_unique()} weeks × {pooled[LOCATION].n_unique()} locations\n")
pooled

### BR plus each named UF, from one article

The spread is visible on a single fixture page: one article becomes a national row
and one row per state it names, all in the same week.


In [ ]:
example = pooled.filter(pl.col(EW) == pooled[EW].min())
print(f"week {example[EW][0]}: {example.height} rows from "
      f"{example[N_ARTICLES].max()} article(s)")
example.select(LOCATION, N_ARTICLES, EMB_STD)

## Integrity checks — fixture tables

In [ ]:
for check in embedding_checks(articles, pooled, FU_PATH):
    print(check)

## Optional — the real corpus

The fixture is five pages. Truncation at scale, the weekly coverage and the effect
of the Spanish slice only exist in the 4,569-article corpus:

```bash
uv sync --extra embeddings
uv run arboili transform news-embeddings
```


In [ ]:
if POOLED_PATH.exists():
    real_articles = pl.read_parquet(ARTICLES_PATH)
    real_pooled = pl.read_parquet(POOLED_PATH)

    print(f"{real_articles.height:,} articles → {real_pooled.height:,} pooled rows")
    print(f"weeks     {real_pooled[EW].min()} → {real_pooled[EW].max()}")
    print(f"locations {real_pooled[LOCATION].n_unique()} "
          f"(BR + {real_pooled[LOCATION].n_unique() - 1} UFs)")
    print(f"model     {real_articles['model_id'][0]}@{real_articles['model_revision'][0][:8]}")

    over, longest = emb_steps.truncation_stats(real_articles)
    print(f"\ntruncated {over:,}/{real_articles.height:,} articles "
          f"({100 * over / real_articles.height:.1f}%), longest {longest:,} tokens")
    print("token count percentiles: " + ", ".join(
        f"p{q}={real_articles[N_TOKENS].quantile(q / 100):.0f}" for q in (50, 75, 90, 99)
    ))

    single = real_pooled.filter(pl.col(N_ARTICLES) == 1).height
    print(f"\nrows pooling exactly one article: {single:,}/{real_pooled.height:,} "
          f"({100 * single / real_pooled.height:.1f}%) — `emb_std` is 0 for all of them")

    print("\nweeks per location (BR is the ceiling):")
    print(real_pooled.group_by(LOCATION).agg(pl.len().alias("semanas"))
          .sort("semanas", descending=True).head(8))
else:
    print("Run `uv run arboili transform news-embeddings` first")

In [ ]:
if POOLED_PATH.exists():
    for check in embedding_checks(real_articles, real_pooled, FU_PATH):
        print(check)

## Exploratory — does the embedding track SINAN?

**This is not the protocol-00 comparison.** It is a first look at the paper's
actual question, and the caveats are the same ones braço 01's notebook carries:
coverage *follows* the epidemic curve, the news series is national while the target
is UF, and per-UF rows are mention-based.

A 768-wide vector cannot be correlated with a scalar curve, so it is reduced two
ways and both are reported, because they answer different questions:

- **PC1** — the leading principal component of the national weekly means, fit by
  SVD on the series itself. The share of variance it explains is printed: a PC1
  that explains 12% is not a summary of anything, and quoting its correlation
  without that number would be a category error.
- **`emb_std`** — the dispersion inside the week, which needs no projection at all.

SINAN here is `ew_symptom_onset`, the same Sunday calendar, summed to (week, state).


### What `pc1` and `emb_std` actually measure

Neither is an **evaluation metric**. They are two candidate *features* — inputs a
forecaster would consume — and the Spearman coefficients below are a first look,
not a score. The evaluation is protocol 00 (one model, one metric set, the
shuffled-text negative control), and it has not run.

**`emb_std` is the RMS distance from the week's centroid.** It is built in two
steps: the per-dimension standard deviation across the articles of one
`(ew, location)` cell, then the norm of that 768-vector. Since
`Σ_d var_d = (1/n) Σ_i ‖x_i − x̄‖²`, the norm of the deviation is exactly

```
emb_std = sqrt( (1/n) Σ_i ‖x_i − x̄‖² )
```

the root-mean-square distance from each article to the week's mean. With
L2-normalized vectors that is *angular* spread on the unit sphere, bounded by 2
(every article opposed to every other). High = a week of unrelated stories; low =
one story retold. Three consequences:

- **Zero does not mean "low dispersion", it means "undefined"** for the 59.8% of
  rows that pool a single article — there is nothing there to disagree with.
- **The scale carries `dim`.** There is no division by `√768`, so the values are
  not comparable across encoders of different widths.
- **It is confounded with volume.** More articles are more chances to disagree,
  which is why +0.233 here and `n_articles`' +0.347 below are partly one signal.

**`pc1` is the leading direction of the weekly means, fit in-sample.** A 768-vector
cannot be correlated with a scalar curve, so the national means are centred and
decomposed by SVD: the first right-singular vector is the direction along which
the weeks vary most, and a week's score is its projection onto it. The share
printed beside it is `s₁² / Σs²`, and it is not decoration — at **20.8%** the
direction explains a fifth of its own series, so the +0.119 below has to be read
against that number rather than on its own. Three more consequences:

- **The sign is arbitrary.** SVD returns the direction up to a sign, and flipping
  it flips the coefficient to −0.119. Nothing in the cell below fixes which of the
  two is reported.
- **It is fit on the whole window**, so the direction has already seen the weeks it
  is being read against. That is acceptable for a first look and is precisely why
  this is "for reading, not for feeding".
- **It is national and contemporaneous**, like every series here — the lead/lag
  scan below is what tests whether that matters.

In [ ]:
sinan_weekly = (
    pl.scan_parquet(SINAN_PATH)
    .group_by("ew_symptom_onset", "state_abbrev")
    .agg(pl.col("case_count").sum().alias("casos"))
    .collect()
)

if POOLED_PATH.exists():
    br = (
        real_pooled.filter(pl.col(LOCATION) == "BR")
        .join(
            sinan_weekly.group_by("ew_symptom_onset")
            .agg(pl.col("casos").sum())
            .rename({"ew_symptom_onset": EW}),
            on=EW,
            how="inner",
        )
        .sort(EW)
    )

    matrix = br[EMB_MEAN].to_numpy()
    centred = matrix - matrix.mean(axis=0, keepdims=True)
    u, s, _ = np.linalg.svd(centred, full_matrices=False)
    explained = float((s ** 2 / (s ** 2).sum())[0])
    br = br.with_columns(pl.Series("pc1", (u[:, 0] * s[0]).astype(np.float32)))

    print(f"national weeks with both series: {br.height:,}  ({br[EW].min()} → {br[EW].max()})")
    print(f"PC1 explains {100 * explained:.1f}% of the variance of the weekly means\n")
    print(
        pl.DataFrame({
            "series": ["pc1", EMB_STD, "n_articles"],
            "spearman vs weekly cases": [
                br.select(pl.corr(c, "casos", method="spearman")).item()
                for c in ("pc1", EMB_STD, N_ARTICLES)
            ],
        }).with_columns(pl.col("spearman vs weekly cases").round(3))
    )
else:
    print("Run `uv run arboili transform news-embeddings` first")

### Lead or lag?

If the representation carried early warning, its value at week *t* would correlate
better with cases at *t+k* than at *t*. The table shifts both series and reports
the national Spearman coefficient in each direction.


In [ ]:
if POOLED_PATH.exists():
    scan = []
    for k in range(0, 9):
        shifted = br.with_columns(
            pl.col("casos").shift(-k).alias("casos_futuros"),
            pl.col("pc1").shift(-k).alias("pc1_futuro"),
            pl.col(EMB_STD).shift(-k).alias("std_futuro"),
        )
        scan.append({
            "k (weeks)": k,
            "pc1[t] × cases[t+k]": shifted.select(pl.corr("pc1", "casos_futuros", method="spearman")).item(),
            "std[t] × cases[t+k]": shifted.select(pl.corr(EMB_STD, "casos_futuros", method="spearman")).item(),
            "cases[t] × pc1[t+k]": shifted.select(pl.corr("casos", "pc1_futuro", method="spearman")).item(),
        })
    print(pl.DataFrame(scan).with_columns(pl.exclude("k (weeks)").round(3)))
else:
    print("Run `uv run arboili transform news-embeddings` first")

### Per federative unit

Restricted to UFs with at least 20 overlapping weeks, because a coefficient on
eight points is noise. `semanas` is printed next to every number for that reason.

Read these with Step 4 in mind: the scoping is mention-based, so a UF only appears
in the weeks its name appeared in the news, and the weeks with enough articles to
have a dispersion are the busy ones.


In [ ]:
if POOLED_PATH.exists():
    per_uf = (
        real_pooled.filter(pl.col(LOCATION) != "BR")
        .join(
            sinan_weekly.rename({"ew_symptom_onset": EW, "state_abbrev": LOCATION}),
            on=[EW, LOCATION],
            how="inner",
        )
        .sort(EW)
    )
    rows = []
    for (location,), group in per_uf.group_by([LOCATION], maintain_order=True):
        if group.height < 20:
            continue
        rows.append({
            "location": location,
            "semanas": group.height,
            EMB_STD: group.select(pl.corr(EMB_STD, "casos", method="spearman")).item(),
            N_ARTICLES: group.select(pl.corr(N_ARTICLES, "casos", method="spearman")).item(),
        })
    print(pl.DataFrame(rows)
          .with_columns(pl.exclude("location", "semanas").round(3))
          .sort(EMB_STD, descending=True))
else:
    print("Run `uv run arboili transform news-embeddings` first")

## What the exploratory section shows

Run on 2026-10-13, over the 481 national weeks where both series exist
(2013-07-28 → 2026-08-23). Four readings, and the first one is uncomfortable.

- **The vector's *content* adds little.** PC1 explains **20.8%** of the variance
  of the weekly means, and correlates **+0.119** with weekly cases. A leading
  direction that explains a fifth of its own series is not summarising much, and
  its correlation should be read against that, not on its own.
- **What correlates is volume, not meaning.** `n_articles` — braço 01's
  `news_volume` under another name — is the strongest of the three at **+0.347**.
  `emb_std` reaches **+0.233**, and it is partly the same signal: more articles
  means more chances for them to disagree. Neither number is evidence that the
  *embedding* carries something the lexicon does not.
- **`emb_std` is degenerate for most rows.** **59.8%** of the 5,152 pooled rows
  pool exactly one article, so their dispersion is 0 by construction — not
  "low dispersion", but undefined. It behaves like a feature on the busy weeks
  and is a constant on the quiet ones, which is precisely where a covariate is
  usually wanted. This is the strongest argument so far for pooling at a coarser
  grain (month) or against `emb_std` as a feature at all.
- **Contemporaneous, not leading.** `n_articles` peaks at `k = 0` (0.347) and
  decays monotonically to 0.218 at `k = 8`; PC1 is flat around 0.12 in both
  directions. Nothing here warns ahead. That matches braço 01 and the framing
  decision in [`notes/README.md`](../notes/README.md): coverage measures
  attention, not the epidemic's future.

The per-UF coefficients are **stronger where the feature is defined and weaker
where it is not**: DF +0.49 and SC +0.39 for `emb_std`, down through SP +0.31 to
RR −0.14 and RO −0.12. The northern states with negative coefficients are the ones
with the fewest articles per week — the same weeks where `emb_std` is a constant,
so the coefficient is being computed on whatever variation is left. That spread is
the mention-based scoping bias of Step 4 showing up as a number, and it is the
reason these are not 27 independent confirmations of anything.

This is the pattern [Tan 2024](https://arxiv.org/abs/2406.16964) predicts, and the
reason protocol 00 mandates the negative control: a rich text representation that
correlates no better than the count of articles in the week is not earning its
768 dimensions. Whether it earns them under a forecaster is still open.

None of this is the protocol-00 comparison. That comparison — braços 01, 02 and 03
under one model and one set of metrics, with the negative control — has not run,
and these numbers must not be quoted as its result.


## Running it for real

This notebook mirrors `transform()` in
`src/pipeline/news/embeddings/transform.py`:

```bash
uv sync --extra embeddings                  # torch + transformers (~800 MB), once
uv run arboili transform news-embeddings    # 4,569 articles → both tables
```

Writes `data/pipeline/news/EBC_dengue_embeddings_article.parquet` (one row per
article — the expensive half) and `..._embeddings_EW.parquet` (the pooled weekly
table the forecaster consumes). Parquet only: a 768-float column does not survive
csv text form.

Re-pooling from the article table is seconds. Re-encoding is the whole run, which
is why both are written.

## What this notebook does not do

- **No fine-tuning.** The encoder is frozen, by the design `notes/02` fixes.
  Fine-tuning on relevance classification is a separate sub-experiment and would
  change the braço's nature — decide it before comparing, not during.
- **No comparison against braços 01 and 03.** They publish the same grain so the
  comparison is possible; running it is protocol 00, and it has not run.
- **No negative control.** Shuffling the text and re-running is what protocol 00
  requires to prove the representation is being used at all — Tan 2024
  (arXiv:2406.16964) is the reason that control exists.
- **No dimensionality reduction for the forecaster.** PC1 above is for reading,
  not for feeding; whether the 768-wide vector is used whole or reduced (and the
  variance that keeps) is a modelling decision still open in `notes/02`.
- **No chunked pooling.** Long articles are truncated at 512 tokens. Whether the
  tail matters is an open question this notebook measures the size of, not one it
  answers.
